In [0]:
from pyspark.sql.functions import *

In [0]:
control_col = ['load_dt','load_dttm']
shop_name_df = spark.table("retail_sales_dw.shop_name_silver").drop(*control_col)
fact_sales_df = spark.table("retail_sales_dw.fact_sales_silver").drop(*control_col)

In [0]:
final_result_df = (
    fact_sales_df
    .join(shop_name_df, ["shop_id"], "left")
    .groupBy("sales_date")
    .agg(
        floor(sum("sales_amt")).alias("sum_sales"),
        floor(avg("sales_amt")).alias("avg_sales"),
        count("sales_amt").alias("bucket_count"),
        max("sales_amt").alias("max_sales")
    )
)
final_result_df.display()

In [0]:
# select top 5 date6
final_result_df.orderBy("sum_sales").limit(5).display()

In [0]:
daily_kpi_df = (
    fact_sales_df
    .groupBy("sales_date")
    .agg(
        round(sum("sales_amt"), 2).alias("total_sales"),
        sum("sales_qty").alias("total_quantity"),
        count("transaction_id").alias("total_transactions"),
        round(avg("sales_amt"), 2).alias("avg_transaction_value"),
        round(avg("sales_qty"), 2).alias("avg_quantity_per_transaction"),
        round(max("sales_amt"), 2).alias("max_transaction_value"),
        round(min("sales_amt"), 2).alias("min_transaction_value")
    )
)
daily_kpi_df.display()

(
    daily_kpi_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dw.fact_sales_daily_kpi")
)

In [0]:
shop_daily_df = (
    fact_sales_df
    .join(shop_name_df, ["shop_id"], "left")
    .groupBy(
        "sales_date",
        "shop_id",
        "shop_name",
        "branch_name"
    )
    .agg(
        round(sum("sales_amt"), 2).alias("total_sales"),
        sum("sales_qty").alias("total_quantity"),
        count("transaction_id").alias("total_transactions"),
        round(avg("sales_amt"), 2).alias("avg_transaction_value")
    )
)
shop_daily_df.display()

(
    shop_daily_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dw.fact_sales_shop_daily")
)

In [0]:
monthly_kpi_df = (
    fact_sales_df
    .withColumn("sales_month", date_format("sales_date", "yyyy-MM"))
    .groupBy("sales_month")
    .agg(
        round(sum("sales_amt"), 2).alias("total_sales"),
        sum("sales_qty").alias("total_quantity"),
        count("transaction_id").alias("total_transactions"),
        round(avg("sales_amt"), 2).alias("avg_transaction_value")
    )
)
monthly_kpi_df.display()

(
    monthly_kpi_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dw.fact_sales_monthly_kpi")
)

In [0]:
shop_monthly_df = (
    fact_sales_df
    .join(shop_name_df, ["shop_id"], "left")
    .withColumn("sales_month", date_format("sales_date", "yyyy-MM"))
    .groupBy(
        "sales_month",
        "shop_id",
        "shop_name",
        "branch_name"
    )
    .agg(
        round(sum("sales_amt"), 2).alias("total_sales"),
        sum("sales_qty").alias("total_quantity"),
        count("transaction_id").alias("total_transactions"),
        round(avg("sales_amt"), 2).alias("avg_transaction_value")
    )
)
shop_monthly_df.display()

(
    shop_monthly_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dw.fact_sales_shop_monthly")
)

In [0]:
kpi_summary_df = (
    fact_sales_df
    .agg(
        round(sum("sales_amt"), 2).alias("total_sales"),
        sum("sales_qty").alias("total_quantity"),
        count("transaction_id").alias("total_transactions"),
        countDistinct("shop_id").alias("active_shops"),
        round(avg("sales_amt"), 2).alias("avg_transaction_value"),
        round(max("sales_amt"), 2).alias("max_transaction_value"),
        round(min("sales_amt"), 2).alias("min_transaction_value")
    )
)
kpi_summary_df.display()

(
    kpi_summary_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_sales_dw.fact_sales_kpi_summary")
)